In [1]:
import torch
import torch.nn as nn

class TransformerFake(nn.Module):
    def __init__(self, config):
        super().__init__()
        
    def forward(self, x):
        return x 

In [2]:
class LayerNormalizationFake(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        
    def forward(self, x):
        return x 

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_of_heads": 12,
    "num_of_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False    
}

In [4]:
class GPTModelFake(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        self.token_embedding = nn.Embedding( config['vocab_size'] , config['embedding_dim'] )
        self.positional_embedding = nn.Embedding( config['context_length'] , config['embedding_dim'] )
        self.dropout_embedding = nn.Dropout( config['drop_rate'])
        
        self.transformer_block = nn.Sequential(
            *[TransformerFake(config) for _ in range(config['num_of_layers'])]
        )
        self.final_normalization = LayerNormalizationFake(config['embedding_dim'])
        self.output_head = nn.Linear(config['embedding_dim'] , config['vocab_size'])
        
    def forward(self, inputs):
        batch_size , no_of_token = inputs.shape
        token_embedding = self.token_embedding(inputs)
        positional_embedding = self.positional_embedding(
            torch.arange(no_of_token, device=inputs.device)
        )
        
        x = token_embedding + positional_embedding
        x = self.dropout_embedding(x)
        x = self.transformer_block(x)
        x = self.final_normalization(x)
        logits = self.output_head(x)
        return logits

In [5]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

batch = []

tx1 = "Every efforts moves you"
tx2 = "Everyday hold a"

batch.append(torch.tensor(tokenizer.encode(tx1)))
batch.append(torch.tensor(tokenizer.encode(tx2)))
batch = torch.stack(batch, dim=0)
batch

tensor([[6109, 4040, 6100,  345],
        [6109,  820, 1745,  257]])

In [6]:
batch.shape

torch.Size([2, 4])

In [7]:
torch.manual_seed(123)

model = GPTModelFake(GPT_CONFIG_124M)
logits = model(batch)
logits

tensor([[[-1.0612e+00,  1.7684e-01, -8.5291e-01,  ..., -9.6077e-01,
          -1.8939e-01, -4.4224e-01],
         [-4.0205e-01,  1.2117e+00, -5.9921e-02,  ..., -8.4721e-01,
           1.4017e+00,  1.1343e+00],
         [ 8.3783e-01,  1.4459e+00, -1.9350e-01,  ...,  1.4067e+00,
          -1.8978e-01,  6.5526e-01],
         [ 3.6188e-01,  1.7019e+00, -7.6544e-01,  ...,  1.0078e+00,
           1.7372e-01, -5.0901e-01]],

        [[-9.2914e-01, -7.6811e-02, -1.1879e+00,  ..., -1.7291e+00,
          -1.6213e-02, -5.4730e-01],
         [-1.0614e+00, -2.3869e-01, -9.0760e-04,  ..., -3.8184e-01,
           3.1712e-01,  8.6845e-01],
         [ 3.5039e-01,  8.0706e-01,  2.2796e-01,  ...,  2.1755e-01,
           5.0086e-01, -9.8705e-01],
         [ 8.0212e-01, -6.5563e-01,  5.6202e-01,  ...,  1.9586e+00,
          -1.3100e-01,  3.2404e-01]]], grad_fn=<ViewBackward0>)

In [8]:
logits.shape   # vocab size

torch.Size([2, 4, 50257])